# 🌦️ Indonesia Weather ETL Pipeline
### OpenMeteo API → BigQuery → Looker Studio

**Course project — Purwadhika DS & ML Program**

---

## 🗺️ Alur Pipeline

```
STEP 0  Setup & Konfigurasi
  │
STEP 1  EXTRACT   ── Ambil data cuaca dari OpenMeteo API
  │                   10 kota Indonesia × 30 hari × 11 variabel
  │
STEP 2  TRANSFORM ── Bersihkan & perkaya data
  │                   9 task transformasi
  │
STEP 3  LOAD      ── Upload ke BigQuery
  │                   1 fact table + 2 dimension tables
  │
STEP 4  ANALYZE   ── Query analitik di BigQuery
  │                   6 query untuk Looker Studio
  │
STEP 5  VISUALIZE ── Panduan koneksi Looker Studio
                      Layout dashboard + KPI cards
```

## 🎯 Pertanyaan Bisnis yang Ingin Dijawab

1. Kota mana yang **paling panas** dan **paling hujan** pada periode 20 Sep – 19 Okt 2026?
2. Bagaimana **tren suhu harian** untuk setiap kota?
3. Berapa **total hari hujan** dan distribusi **intensitas hujan**?
4. Kapan **waktu terpanas dalam seminggu** (hari apa)?
5. Apakah ada kota yang mengalami **suhu ekstrem / anomali**?
6. Bagaimana **perbandingan UV Index** antar kota?

---
> **Reproducibility:** data dibekukan sebagai snapshot (`data/dataset_raw.csv`, periode 20 Sep – 19 Okt 2026) agar angka sama dengan dashboard. Lihat flag `USE_SNAPSHOT` dan `LOAD_TO_BQ` di Step 0.

## 📝 Konteks, Koreksi & Keterbatasan

**Konteks:** notebook ini dikerjakan sebagai course project Purwadhika (template tugas dengan skema fact/dim yang sudah ditentukan). Kode extract, transform, load, view, query analitik, dan dashboard Looker Studio dikerjakan sendiri.

| Area | Keputusan di notebook ini |
|---|---|
| `is_weekend` | `>= 5` → Sabtu & Minggu |
| `season` | `Dry` (Apr–Sep) / `Wet` (Okt–Mar), sesuai pola musim Indonesia |
| `uv_category` | Skala WHO (UV desimal): Low <3, Moderate 3–<6, High 6–<8, Very High 8–<11, Extreme ≥11 |
| Tanggal | Dikunci 2026-09-20 s/d 2026-10-19 + snapshot CSV |

**Keterbatasan**
- Data 30 hari × 10 kota (300 baris) dari endpoint *forecast* Open-Meteo: data model, bukan pengukuran stasiun. Nilai untuk hari terbaru dapat direvisi API, sehingga menarik ulang dari API tidak dijamin menghasilkan angka yang sama (karena itu dipakai snapshot).
- `humidity_pct` = kelembaban **maksimum** harian; heat index memakai nilai ini dengan suhu rata-rata, sehingga cenderung overestimate.
- `is_anomaly_temp` hanya menandai anomali panas (> mean + 2σ per kota).
- `is_rainy` = `rain_mm > 0` (termasuk hujan sangat tipis, 0.1 mm); akibatnya Medan tercatat hujan 30 dari 30 hari.
- Pipeline dijalankan manual (tanpa scheduler); load memakai full replace (idempotent).

---
## ⚙️ STEP 0 — Setup & Konfigurasi

In [27]:
# Import semua library yang dibutuhkan
import os
import requests
import pandas as pd
import numpy as np
from datetime import date, timedelta
import pandas_gbq
from google.cloud import bigquery
from google.oauth2 import service_account
from dotenv import load_dotenv, find_dotenv

# Baca file .env
load_dotenv(find_dotenv(usecwd=True))

def get_secret(key: str) -> str:
    value = os.environ.get(key)
    if not value:
        raise KeyError(f"'{key}' belum diset. Buat file .env di folder notebook (lihat .env.example), lalu restart kernel.")
    return value

print('Imports OK')

Imports OK


In [28]:
from pathlib import Path

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# KONFIGURASI PIPELINE (boleh diubah)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Tanggal dikunci (snapshot 20 Sep - 19 Okt 2026).
# Catatan: endpoint forecast Open-Meteo dapat merevisi nilai terbaru, jadi menarik ulang dari API
# TIDAK dijamin menghasilkan angka yang sama dengan dashboard. Untuk data terbaru: END_DATE = date.today()
END_DATE   = date(2026, 10, 19)
START_DATE = END_DATE - timedelta(days=29)   # 30 hari

# Sumber data
USE_SNAPSHOT  = True                              # True: baca data/dataset_raw.csv (angka = dashboard). False: tarik ulang dari API.
SNAPSHOT_PATH = Path('../data/dataset_raw.csv')

# Proteksi BigQuery: biarkan False agar tabel/view yang dipakai dashboard tidak tertimpa tanpa sengaja.
LOAD_TO_BQ    = False

# 10 Kota besar Indonesia
# Format: city_id, nama tampilan, latitude, longitude, provinsi
CITIES = [
    {'city_id': 'jakarta',    'name': 'Jakarta',    'lat': -6.2088,  'lon': 106.8456, 'province': 'DKI Jakarta',         'island': 'Jawa'},
    {'city_id': 'surabaya',   'name': 'Surabaya',   'lat': -7.2575,  'lon': 112.7521, 'province': 'Jawa Timur',          'island': 'Jawa'},
    {'city_id': 'bandung',    'name': 'Bandung',    'lat': -6.9175,  'lon': 107.6191, 'province': 'Jawa Barat',          'island': 'Jawa'},
    {'city_id': 'medan',      'name': 'Medan',      'lat':  3.5952,  'lon':  98.6722, 'province': 'Sumatera Utara',      'island': 'Sumatera'},
    {'city_id': 'makassar',   'name': 'Makassar',   'lat': -5.1477,  'lon': 119.4327, 'province': 'Sulawesi Selatan',   'island': 'Sulawesi'},
    {'city_id': 'semarang',   'name': 'Semarang',   'lat': -6.9932,  'lon': 110.4203, 'province': 'Jawa Tengah',        'island': 'Jawa'},
    {'city_id': 'palembang',  'name': 'Palembang',  'lat': -2.9761,  'lon': 104.7754, 'province': 'Sumatera Selatan',   'island': 'Sumatera'},
    {'city_id': 'denpasar',   'name': 'Denpasar',   'lat': -8.6705,  'lon': 115.2126, 'province': 'Bali',               'island': 'Bali'},
    {'city_id': 'yogyakarta', 'name': 'Yogyakarta', 'lat': -7.7972,  'lon': 110.3688, 'province': 'DIY',                'island': 'Jawa'},
    {'city_id': 'balikpapan', 'name': 'Balikpapan', 'lat': -1.2379,  'lon': 116.8529, 'province': 'Kalimantan Timur',  'island': 'Kalimantan'},
]

print('Config OK')
print(f'   Periode  : {START_DATE} s/d {END_DATE} ({(END_DATE - START_DATE).days + 1} hari)')
print(f'   Kota     : {len(CITIES)} kota')
print(f'   Target   : {len(CITIES) * ((END_DATE - START_DATE).days + 1)} baris data')
print(f'   Sumber   : {"snapshot CSV" if USE_SNAPSHOT else "Open-Meteo API"} | Load ke BigQuery: {LOAD_TO_BQ}')

Config OK
   Periode  : 2026-09-20 s/d 2026-10-19 (30 hari)
   Kota     : 10 kota
   Target   : 300 baris data
   Sumber   : snapshot CSV | Load ke BigQuery: False


---
## 📡 STEP 1 — EXTRACT: Ambil Data dari OpenMeteo API

### Tentang OpenMeteo
- **URL**: https://api.open-meteo.com/v1/forecast
- **API Docs**: https://open-meteo.com/en/docs 
- **Auth**: ❌ Tidak perlu API key
- **Limit**: Bebas untuk penggunaan non-komersial
- **Data**: Historis hingga 92 hari ke belakang + forecast 16 hari ke depan

### Variabel yang Diambil (Daily)

| No | Variabel API | Kolom Output | Satuan | Keterangan |
|---|---|---|---|---|
| 1 | `temperature_2m_max` | `temp_max_c` | °C | Suhu tertinggi hari itu |
| 2 | `temperature_2m_min` | `temp_min_c` | °C | Suhu terendah hari itu |
| 3 | `temperature_2m_mean` | `temp_mean_c` | °C | Rata-rata suhu harian |
| 4 | `precipitation_sum` | `precipitation_mm` | mm | Total curah hujan (hujan + salju) |
| 5 | `rain_sum` | `rain_mm` | mm | Curah hujan saja |
| 6 | `wind_speed_10m_max` | `wind_speed_kmh` | km/h | Kecepatan angin tertinggi |
| 7 | `wind_gusts_10m_max` | `wind_gust_kmh` | km/h | Gusts angin tertinggi |
| 8 | `relative_humidity_2m_max` | `humidity_pct` | % | Kelembaban relatif maksimum |
| 9 | `uv_index_max` | `uv_index` | 0–11+ | UV Index tertinggi |
| 10 | `weather_code` | `weather_code` | kode | Kondisi cuaca (WMO standard) |
| 11 | `sunshine_duration` | `sunshine_sec` | detik | Durasi sinar matahari |

### Struktur Response API
```json
{
  "latitude": -6.2088,
  "longitude": 106.8456,
  "timezone": "Asia/Jakarta",
  "daily": {
    "time":                    ["2024-01-01", "2024-01-02", ...],
    "temperature_2m_max":      [32.1, 33.4, ...],
    "temperature_2m_min":      [24.5, 25.1, ...],
    "temperature_2m_mean":     [28.3, 29.2, ...],
    "precipitation_sum":       [12.4, 0.0, ...],
    "rain_sum":                [12.4, 0.0, ...],
    "wind_speed_10m_max":      [18.2, 22.1, ...],
    "wind_gusts_10m_max":      [32.4, 28.0, ...],
    "relative_humidity_2m_max":[88, 75, ...],
    "uv_index_max":            [8.5, 9.2, ...],
    "weather_code":            [61, 0, ...],
    "sunshine_duration":       [18000, 32400, ...]
  }
}
```

In [29]:
# ─────────────────────────────────────────────
# TASK 1.1 — Fungsi extract SATU kota
# ─────────────────────────────────────────────
# Buat fungsi yang hit API untuk satu kota dan satu rentang tanggal.
# Return: DataFrame dengan satu baris per hari.

# WMO Weather Code → label yang mudah dibaca
WMO_LABELS = {
    0: 'Clear Sky',        1: 'Mainly Clear',     2: 'Partly Cloudy',  3: 'Overcast',
    45: 'Fog',             48: 'Icy Fog',
    51: 'Light Drizzle',   53: 'Drizzle',         55: 'Heavy Drizzle',
    61: 'Light Rain',      63: 'Moderate Rain',   65: 'Heavy Rain',
    80: 'Light Showers',   81: 'Showers',         82: 'Heavy Showers',
    95: 'Thunderstorm',    96: 'Thunderstorm+Hail', 99: 'Heavy Thunderstorm'
}

def extract_one_city(city: dict, start: date, end: date) -> pd.DataFrame:
    """
    Ambil data cuaca harian untuk satu kota dari OpenMeteo API.

    Parameters
    ----------
    city  : dict dengan keys → city_id, name, lat, lon, province, island
    start : tanggal mulai
    end   : tanggal akhir

    Returns
    -------
    pd.DataFrame — satu baris per hari dengan kolom:
    city_id, city_name, province, island, date,
    temp_max_c, temp_min_c, temp_mean_c,
    precipitation_mm, rain_mm,
    wind_speed_kmh, wind_gust_kmh,
    humidity_pct, uv_index, weather_code, sunshine_sec,
    fetched_at
    """

    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": city['lat'],
        "longitude": city['lon'],
        "start_date": start.strftime('%Y-%m-%d'),
        "end_date": end.strftime('%Y-%m-%d'),
        "timezone": "Asia/Jakarta",
        "daily": [
            "temperature_2m_max", 
            "temperature_2m_min", 
            "temperature_2m_mean",
            "precipitation_sum", 
            "rain_sum", 
            "wind_speed_10m_max", 
            "wind_gusts_10m_max", 
            "relative_humidity_2m_max", 
            "uv_index_max", 
            "weather_code", 
            "sunshine_duration"
        ],

    }

    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        print(f"error getting {city['name']}: {response.status_code}")
        return pd.DataFrame()
        
    data = response.json()
    df = pd.DataFrame(data['daily'])

    rename_mapping = {
        'time': 'date',
        'temperature_2m_max': 'temp_max_c',
        'temperature_2m_min': 'temp_min_c',
        'temperature_2m_mean': 'temp_mean_c',
        'precipitation_sum': 'precipitation_mm',
        'rain_sum': 'rain_mm',
        'wind_speed_10m_max': 'wind_speed_kmh',
        'wind_gusts_10m_max': 'wind_gust_kmh',
        'relative_humidity_2m_max': 'humidity_pct',
        'uv_index_max': 'uv_index',
        'weather_code': 'weather_code',
        'sunshine_duration': 'sunshine_sec'
    }
    df = df.rename(columns=rename_mapping)
    
    #add column
    df['city_id'] = city['city_id']
    df['city_name'] = city['name']
    df['province'] = city['province']
    df['island'] = city['island']
    df['fetched_at'] = pd.Timestamp.utcnow()
    
    #order
    ordered_columns = [
        'city_id', 'city_name', 'province', 'island', 'date',
        'temp_max_c', 'temp_min_c', 'temp_mean_c',
        'precipitation_mm', 'rain_mm',
        'wind_speed_kmh', 'wind_gust_kmh',
        'humidity_pct', 'uv_index', 'weather_code', 'sunshine_sec',
        'fetched_at'
    ]

    return df[ordered_columns]


if not USE_SNAPSHOT:   # tes koneksi API hanya relevan jika data ditarik ulang
    # ── TEST: extract satu kota (Jakarta, 3 hari) ──
    df_test = extract_one_city(CITIES[0], END_DATE - timedelta(days=2), END_DATE)

    print(f'Shape   : {df_test.shape}')       # harus (3, 17)
    print(f'Kolom   : {list(df_test.columns)}')
    print(f'Tanggal : {df_test["date"].tolist()}')
    df_test


In [30]:
# ─────────────────────────────────────────────
# TASK 1.2 — Loop semua kota, gabung jadi satu DataFrame
# ─────────────────────────────────────────────
# Panggil extract_one_city() untuk setiap kota di CITIES.
# Simpan hasilnya ke df_raw.

def extract_all_cities(cities, start, end):
    all_data = []
    for city in cities:
        df_city = extract_one_city(city, start, end)
        all_data.append(df_city)
    df_raw = pd.concat(
        all_data,
        ignore_index=True
    )
    
    return df_raw

if USE_SNAPSHOT and SNAPSHOT_PATH.exists():
    print(f'Membaca snapshot: {SNAPSHOT_PATH}')
    df_raw = pd.read_csv(SNAPSHOT_PATH)
    df_raw['fetched_at'] = pd.to_datetime(df_raw['fetched_at'], utc=True)
    DATA_SOURCE = 'snapshot'
else:
    if USE_SNAPSHOT:
        print(f'Snapshot tidak ditemukan di {SNAPSHOT_PATH} -> menarik dari API')
    print(f'Mengambil data {len(CITIES)} kota x {(END_DATE-START_DATE).days+1} hari ...')
    df_raw = extract_all_cities(CITIES, START_DATE, END_DATE)
    DATA_SOURCE = 'api'

print(f'\n✅ Extract selesai!')
print(f'   Total baris : {len(df_raw)}')
print(f'   Jumlah kota : {df_raw["city_id"].nunique()}')
print(f'   Rentang tgl : {df_raw["date"].min()} → {df_raw["date"].max()}')
df_raw.head()

Membaca snapshot: ../data/dataset_raw.csv

✅ Extract selesai!
   Total baris : 300
   Jumlah kota : 10
   Rentang tgl : 2026-09-20 → 2026-10-19


,city_id,city_name,province,island,date,temp_max_c,temp_min_c,temp_mean_c,precipitation_mm,rain_mm,wind_speed_kmh,wind_gust_kmh,humidity_pct,uv_index,weather_code,sunshine_sec,fetched_at
0,jakarta,Jakarta,DKI Jakarta,Jawa,2026-09-20,33.8,25.2,28.7,0.5,0.5,12.5,29.9,93,9.15,51,40313.93,2026-10-19 15:14:53.018801+00:00
1,jakarta,Jakarta,DKI Jakarta,Jawa,2026-09-21,34.2,22.6,28.6,22.7,8.0,11.7,33.8,100,9.10,82,36487.54,2026-10-19 15:14:53.018801+00:00
2,jakarta,Jakarta,DKI Jakarta,Jawa,2026-09-22,35.1,25.8,29.8,1.4,0.8,11.8,34.9,83,9.10,53,39861.39,2026-10-19 15:14:53.018801+00:00
3,jakarta,Jakarta,DKI Jakarta,Jawa,2026-09-23,34.5,25.8,29.5,0.4,0.4,13.7,36.0,86,9.15,51,40560.53,2026-10-19 15:14:53.018801+00:00
4,jakarta,Jakarta,DKI Jakarta,Jawa,2026-09-24,34.9,26.4,29.8,2.0,0.9,11.8,33.5,83,9.00,80,40630.95,2026-10-19 15:14:53.018801+00:00


In [31]:
# ─────────────────────────────────────────────
# TASK 1.3 — Cek raw data sebelum transform
# ─────────────────────────────────────────────

print('=== Info DataFrame ===')
display(df_raw.info())

print('\n=== Null Values per Kolom ===')
display(df_raw.isnull().sum())

print('\n=== Statistik Deskriptif (kolom numerik) ===')
display(df_raw.describe())

=== Info DataFrame ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   city_id           300 non-null    object             
 1   city_name         300 non-null    object             
 2   province          300 non-null    object             
 3   island            300 non-null    object             
 4   date              300 non-null    object             
 5   temp_max_c        300 non-null    float64            
 6   temp_min_c        300 non-null    float64            
 7   temp_mean_c       300 non-null    float64            
 8   precipitation_mm  300 non-null    float64            
 9   rain_mm           300 non-null    float64            
 10  wind_speed_kmh    300 non-null    float64            
 11  wind_gust_kmh     300 non-null    float64            
 12  humidity_pct      300 non-null    int64  

None


=== Null Values per Kolom ===


city_id             0
city_name           0
province            0
island              0
date                0
temp_max_c          0
temp_min_c          0
temp_mean_c         0
precipitation_mm    0
rain_mm             0
wind_speed_kmh      0
wind_gust_kmh       0
humidity_pct        0
uv_index            0
weather_code        0
sunshine_sec        0
fetched_at          0
dtype: int64


=== Statistik Deskriptif (kolom numerik) ===


,temp_max_c,temp_min_c,temp_mean_c,precipitation_mm,rain_mm,wind_speed_kmh,wind_gust_kmh,humidity_pct,uv_index,weather_code,sunshine_sec
count,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000
mean,33.173667,24.418333,28.379667,2.765000,1.251333,15.038333,38.497000,86.930000,8.981833,43.076667,39464.942767
std,2.162111,1.910146,1.566226,6.109886,2.390372,3.875594,8.064138,8.731891,0.797233,29.610811,5996.862107
min,27.600000,19.400000,24.200000,0.000000,0.000000,5.900000,19.400000,65.000000,2.250000,0.000000,1269.870000
25%,31.600000,23.400000,26.975000,0.000000,0.000000,12.275000,32.700000,80.000000,8.950000,3.000000,39682.915000
50%,33.200000,24.900000,28.800000,0.500000,0.300000,15.200000,38.700000,87.000000,9.150000,51.000000,40700.575000
75%,34.900000,25.725000,29.700000,2.000000,1.300000,17.725000,44.300000,95.000000,9.300000,53.000000,42595.937500
max,38.100000,27.600000,31.100000,40.400000,19.300000,23.700000,58.700000,100.000000,9.650000,95.000000,43200.000000


In [32]:
# Simpan snapshot hanya jika data baru ditarik dari API dan belum ada snapshot
# (snapshot yang dipakai dashboard tidak boleh tertimpa).
if DATA_SOURCE == 'api' and not SNAPSHOT_PATH.exists():
    SNAPSHOT_PATH.parent.mkdir(parents=True, exist_ok=True)
    df_raw.to_csv(SNAPSHOT_PATH, index=False)
    print(f'Snapshot disimpan: {SNAPSHOT_PATH}')
else:
    print(f'Snapshot tidak ditulis ulang (sumber data: {DATA_SOURCE}).')


Snapshot tidak ditulis ulang (sumber data: snapshot).


---
## 🔧 STEP 2 — TRANSFORM: Bersihkan & Perkaya Data

Ada **9 task transformasi** yang perlu dilakukan secara berurutan.

| # | Task | Tujuan |
|---|---|---|
| T1 | Fix tipe data | date → datetime, numerik konsisten |
| T2 | Handle null values | isi nilai yang hilang dengan logika yang tepat |
| T3 | Validasi range | pastikan nilai dalam batas fisik yang masuk akal |
| T4 | Derived: dimensi waktu | tahun, bulan, minggu, hari — untuk filter Looker |
| T5 | Derived: sunshine hours | konversi detik → jam, lebih mudah diinterpretasi |
| T6 | Derived: heat index | "suhu yang dirasakan" berdasarkan suhu + kelembaban |
| T7 | Derived: kategori cuaca | label teks dari WMO code + kategorisasi hujan/suhu |
| T8 | Derived: flag & anomali | is_rainy, is_extreme_heat, is_anomaly |
| T9 | Deduplikasi & final check | pastikan tidak ada baris ganda |

In [33]:
# ─────────────────────────────────────────────
# T1 — Fix Tipe Data
# ─────────────────────────────────────────────
# Tujuan: pastikan setiap kolom punya tipe data yang benar.
#
# Yang perlu dilakukan:
#   • 'date'                                → pd.to_datetime(), lalu ambil .dt.date (bukan datetime)
#   • Kolom numerik (temp, rain, wind, dll) → pd.to_numeric(errors='coerce')
#   • 'weather_code'                        → Int64 (nullable integer, bukan float)

df = df_raw.copy()

# Fix date
df['date'] = pd.to_datetime(df['date']).dt.date

# Fix numerik
df['temp_max_c'] = pd.to_numeric(df['temp_max_c'], errors='coerce')
df['temp_min_c'] = pd.to_numeric(df['temp_min_c'], errors='coerce')
df['temp_mean_c'] = pd.to_numeric(df['temp_mean_c'], errors='coerce')
df['precipitation_mm'] = pd.to_numeric(df['precipitation_mm'], errors='coerce')
df['rain_mm'] = pd.to_numeric(df['rain_mm'], errors='coerce')
df['wind_speed_kmh'] = pd.to_numeric(df['wind_speed_kmh'], errors='coerce')
df['wind_gust_kmh'] = pd.to_numeric(df['wind_gust_kmh'], errors='coerce')
df['humidity_pct'] = pd.to_numeric(df['humidity_pct'], errors='coerce')
df['sunshine_sec'] = pd.to_numeric(df['sunshine_sec'], errors='coerce')

# Fix weather_code
df['weather_code'] = df['weather_code'].astype('Int64')

print('T1 selesai — tipe data diperbaiki')
df.dtypes

T1 selesai — tipe data diperbaiki


city_id                          object
city_name                        object
province                         object
island                           object
date                             object
temp_max_c                      float64
temp_min_c                      float64
temp_mean_c                     float64
precipitation_mm                float64
rain_mm                         float64
wind_speed_kmh                  float64
wind_gust_kmh                   float64
humidity_pct                      int64
uv_index                        float64
weather_code                      Int64
sunshine_sec                    float64
fetched_at          datetime64[ns, UTC]
dtype: object

In [34]:
NUM_COLS = [
    "temp_max_c",
    "temp_min_c",
    "temp_mean_c",
    "precipitation_mm",
    "rain_mm",
    "wind_speed_kmh",
    "wind_gust_kmh",
    "humidity_pct",
    "uv_index",
    "weather_code",
    "sunshine_sec"
]

In [35]:
# ─────────────────────────────────────────────
# T2 — Handle Null Values
# ─────────────────────────────────────────────
# Strategi per kolom:
#
#   rain_mm & precipitation_mm  → fillna(0)
#     Alasan: null berarti tidak ada pengamatan hujan = 0 mm
#
#   sunshine_sec                → fillna(0)
#     Alasan: null pada hari mendung = 0 detik sinar matahari
#
#   uv_index                    → fillna(median per kota)
#     Alasan: UV bervariasi per lokasi, gunakan median kota itu sendiri
#
#   temp_mean_c                 → fillna((temp_max + temp_min) / 2)
#     Alasan: estimasi terbaik jika mean tidak tersedia
#
#   humidity_pct                → fillna(median per kota)
#     Alasan: nilai tipikal kota itu sendiri lebih relevan dari global median

df = df.fillna({
    "rain_mm":0, 
    "precipitation_mm":0, 
    "sunshine_sec":0,
    "temp_mean_c": (df['temp_max_c'] + df['temp_min_c']) / 2
})

df['uv_index'] = df.groupby('city_name')['uv_index'].transform(lambda x: x.fillna(x.median()))

df["humidity_pct"] = df.groupby("city_name")["humidity_pct"].transform(lambda x: x.fillna(x.median()))

print('✅ T2 selesai — null values ditangani')
remaining_nulls = df[NUM_COLS].isnull().sum()
print(f'Sisa null: {remaining_nulls[remaining_nulls > 0].to_dict() or "Tidak ada ✅"}')

✅ T2 selesai — null values ditangani
Sisa null: Tidak ada ✅


In [36]:
# ─────────────────────────────────────────────
# T3 — Validasi Range (Assertions)
# ─────────────────────────────────────────────
# Pastikan semua nilai berada dalam batas fisik yang masuk akal.
# Jika ada yang melanggar, cek sumber datanya.
#
# Batas yang digunakan:
#   Suhu di Indonesia: -5°C s/d 50°C (sangat longgar)
#   Hujan             : >= 0 mm
#   Kelembaban        : 0–105% (toleransi sensor +5)
#   UV index          : 0–16
#   temp_max >= temp_min (hukum fisika)

assert df['temp_max_c'].dropna().between(-5, 50).all(),  'temp_max_c out of range'
assert df['temp_min_c'].dropna().between(-5, 50).all(),  'temp_min_c out of range'
assert (df['temp_max_c'] >= df['temp_min_c']).all(),     'temp_max < temp_min — anomali!'
assert df['rain_mm'].min() >= 0,                         'rain_mm negatif!'
assert df['humidity_pct'].dropna().between(0,105).all(), 'humidity_pct out of range'
assert df['uv_index'].dropna().between(0, 16).all(),     'uv_index out of range'
assert df.duplicated(subset=['city_id','date']).sum() == 0, 'Ada duplikat city+date!'

print('✅ T3 selesai — semua validasi range lulus')

✅ T3 selesai — semua validasi range lulus


In [37]:
# ─────────────────────────────────────────────
# T4 — Derived: Dimensi Waktu
# ─────────────────────────────────────────────
#   year, month, month_name, week, day_name → untuk filter & grouping di Looker Studio
#   is_weekend → True jika Sabtu (dayofweek=5) atau Minggu (dayofweek=6)
#   season     → 'Dry' (Apr-Sep) atau 'Wet' (Okt-Mar)  [pola musim Indonesia, bukan 4 musim]

df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['month_name'] = df['date'].dt.month_name()
df['week'] = df['date'].dt.isocalendar().week
df['day_name'] = df['date'].dt.day_name()
df['is_weekend'] = df['date'].dt.dayofweek >= 5          
df['season'] = np.where(df['date'].dt.month.between(4, 9), 'Dry', 'Wet')

print('✅ T4 selesai — kolom waktu ditambahkan')
print(df[['date','year','month','month_name','week','day_name','is_weekend','season']].head(3).to_string())
print('\nis_weekend:', df['is_weekend'].value_counts().to_dict(), '| season:', df['season'].value_counts().to_dict())

✅ T4 selesai — kolom waktu ditambahkan
        date  year  month month_name  week day_name  is_weekend season
0 2026-09-20  2026      9  September    38   Sunday        True    Dry
1 2026-09-21  2026      9  September    39   Monday       False    Dry
2 2026-09-22  2026      9  September    39  Tuesday       False    Dry

is_weekend: {False: 210, True: 90} | season: {'Wet': 190, 'Dry': 110}


In [38]:
# ─────────────────────────────────────────────
# T5 — Derived: Sunshine Hours & Temp Range
# ─────────────────────────────────────────────
# Dua kalkulasi sederhana yang memperkaya analisis.
#
#   sunshine_hours → sunshine_sec / 3600, dibulatkan 1 desimal
#     Lebih intuitif untuk pembaca dashboard vs detik
#
#   temp_range_c   → temp_max_c - temp_min_c, dibulatkan 1 desimal
#     Indikator variabilitas suhu harian
#     Kota pesisir cenderung range kecil, pegunungan lebih besar

df['sunshine_hours'] = (df['sunshine_sec'] / 3600).round(1)
df['temp_range_c'] = (df['temp_max_c'] - df['temp_min_c']).round(1)

print('✅ T5 selesai')
print(df[['city_name','date','sunshine_sec','sunshine_hours','temp_max_c','temp_min_c','temp_range_c']].head(3).to_string())

✅ T5 selesai
  city_name       date  sunshine_sec  sunshine_hours  temp_max_c  temp_min_c  temp_range_c
0   Jakarta 2026-09-20      40313.93            11.2        33.8        25.2           8.6
1   Jakarta 2026-09-21      36487.54            10.1        34.2        22.6          11.6
2   Jakarta 2026-09-22      39861.39            11.1        35.1        25.8           9.3


In [39]:
# ─────────────────────────────────────────────
# T6 — Derived: Heat Index
# ─────────────────────────────────────────────
# Heat Index = "suhu yang dirasakan tubuh" — kombinasi suhu + kelembaban.
# Semakin lembab, semakin panas yang dirasakan meski suhu sama.
#
# Formula Steadman (simplified, cocok untuk kondisi tropis):
#   HI = -8.78 + (1.61×T) + (2.34×RH) - (0.146×T×RH)
#        - (0.0123×T²) - (0.0164×RH²) + (0.00221×T²×RH)
#        + (0.00723×T×RH²) - (0.000004×T²×RH²)
#
# T  = temp_mean_c
# RH = humidity_pct
#
# Kategori heat_index_c:
#   < 27    → 'Comfortable'
#   27–32   → 'Caution'          (kelelahan mungkin terjadi)
#   32–41   → 'Extreme Caution'  (kram otot, kelelahan panas mungkin)
#   > 41    → 'Danger'           (kemungkinan heat stroke)

# CATATAN: humidity_pct berasal dari relative_humidity_2m_MAX (kelembaban maksimum harian),
# dipasangkan dengan suhu RATA-RATA, sehingga heat index cenderung overestimate.
T  = (df['temp_mean_c'] * 9/5) + 32
RH = df['humidity_pct']

hi_simple = 0.5 * (T + 61.0 + ((T - 68.0) * 1.2) + (RH * 0.094))

c1, c2, c3 = -42.379, 2.04901523, 10.14333127
c4, c5, c6 = -0.22475541, -0.00683783, -0.05481717
c7, c8, c9 = 0.00122874, 0.00085282, -0.00000199

hi_full = (c1 + (c2 * T) + (c3 * RH) + (c4 * T * RH) +
           (c5 * T**2) + (c6 * RH**2) + (c7 * T**2 * RH) +
           (c8 * T * RH**2) + (c9 * T**2 * RH**2))

adj_dry = ((13 - RH) / 4) * ((17 - (T - 95.0).abs()) / 17)**0.5
adj_humid = ((RH - 85) / 10) * ((87 - T) / 5)

hi_full_corrected = np.select(
    condlist=[
        (T >= 80) & (T <= 112) & (RH < 13),
        (T >= 80) & (T <= 87) & (RH > 85)
    ],
    choicelist=[
        hi_full - adj_dry,
        hi_full + adj_humid
    ],
    default=hi_full
)

hi_final_f = np.where(hi_simple < 80, hi_simple, hi_full_corrected)

df["heat_index_c"] = ((hi_final_f - 32) * 5/9).round(1)

def assign_hi_category(HI):
    if HI < 27:
        return 'Comfortable'
    elif HI <= 32:
        return 'Caution'
    elif HI <= 41:
        return 'Extreme Caution'
    else:
        return 'Danger'

df['heat_index_category'] = df['heat_index_c'].apply(assign_hi_category)

print('✅ T6 selesai')
print('Distribusi heat index category:')
print(df['heat_index_category'].value_counts())
df[['city_name','date','temp_mean_c','humidity_pct','heat_index_c','heat_index_category']].head(5)

✅ T6 selesai
Distribusi heat index category:
heat_index_category
Extreme Caution    199
Caution             75
Danger              14
Comfortable         12
Name: count, dtype: int64


,city_name,date,temp_mean_c,humidity_pct,heat_index_c,heat_index_category
0,Jakarta,2026-09-20,28.7,93,37.0,Extreme Caution
1,Jakarta,2026-09-21,28.6,100,38.6,Extreme Caution
2,Jakarta,2026-09-22,29.8,83,37.9,Extreme Caution
3,Jakarta,2026-09-23,29.5,86,37.8,Extreme Caution
4,Jakarta,2026-09-24,29.8,83,37.9,Extreme Caution


In [40]:
# ─────────────────────────────────────────────
# T7 — Derived: Kategori Cuaca
# ─────────────────────────────────────────────
# Tiga kategori turunan untuk memudahkan filter di Looker Studio.
#
# A) weather_label  → teks dari WMO_LABELS dict
#    Contoh: weather_code=63 → 'Moderate Rain'
#
# B) rain_category  → berdasarkan rain_mm
#    0          → 'No Rain'
#    0–5        → 'Light'      (gerimis s/d hujan ringan)
#    5–20       → 'Moderate'   (hujan sedang)
#    20–50      → 'Heavy'      (hujan lebat)
#    > 50       → 'Extreme'    (banjir potensial)
#
# C) uv_category    → berdasarkan uv_index
#    0–2   → 'Low'
#    3–5   → 'Moderate'
#    6–7   → 'High'
#    8–10  → 'Very High'
#    > 10  → 'Extreme'

# A. Weather label dari WMO code
df['weather_label'] = df['weather_code'].map(WMO_LABELS)

# B. Rain category
def assign_rain_category(rain):
    if rain == 0:
        return 'No Rain'
    elif rain <= 5:
        return 'Light'
    elif rain <= 20:
        return 'Moderate'
    elif rain <= 50:
        return 'Heavy'
    else:
        return 'Extreme'

df['rain_category'] = df['rain_mm'].apply(assign_rain_category)

# C. UV category
def assign_uv_category(uv):
    # FIX: sebelumnya `if uv > 0: return 'Low'` sehingga semua nilai berlabel 'Low'
    if uv < 3:
        return 'Low'
    elif uv < 6:
        return 'Moderate'
    elif uv < 8:
        return 'High'
    elif uv < 11:
        return 'Very High'
    else:
        return 'Extreme'

df['uv_category'] = df['uv_index'].apply(assign_uv_category)

print('✅ T7 selesai')
print('\nRain category distribution:')
print((df['rain_category'].value_counts(normalize=True) * 100).round(2))
print('\nTop 5 weather labels:')
print((df['weather_label'].value_counts(normalize=True) * 100).round(2).head())
print('\nUV category distribution:')
print((df['uv_category'].value_counts(normalize=True) * 100).round(2))

✅ T7 selesai

Rain category distribution:
rain_category
Light       58.33
No Rain     36.00
Moderate     5.67
Name: proportion, dtype: float64

Top 5 weather labels:
weather_label
Light Drizzle    32.33
Overcast         24.67
Drizzle          12.67
Light Showers     8.67
Showers           6.67
Name: proportion, dtype: float64

UV category distribution:
uv_category
Very High    95.00
High          3.67
Moderate      0.67
Low           0.67
Name: proportion, dtype: float64


In [41]:
# ─────────────────────────────────────────────
# T8 — Derived: Flag & Anomali
# ─────────────────────────────────────────────
# Flag biner dan deteksi anomali statistik.
#
# A) is_rainy
#    → True jika rain_mm > 0
#    → Digunakan untuk hitung 'jumlah hari hujan'
#
# B) is_extreme_heat
#    → True jika temp_max_c > 37°C
#    → Threshold WHO untuk heat stress
#
# C) is_anomaly_temp
#    → True jika temp_mean_c menyimpang > 2× std dari rata-rata kota itu
#    → Dihitung per kota (BUKAN global)
#    → Digunakan untuk highlight hari tidak normal di time series

# A. Rainy flag
df['is_rainy'] = df['rain_mm'] > 0

# B. Extreme heat flag
df['is_extreme_heat'] = df['temp_max_c'] > 37

# C. Anomali suhu (z-score per kota)
mean_per_city = df.groupby('city_id')['temp_mean_c'].transform('mean')
std_per_city = df.groupby('city_id')['temp_mean_c'].transform('std')

std_per_city = std_per_city.replace(0, np.nan).fillna(1)

# Definisi: HANYA anomali panas (satu sisi, > mean + 2σ per kota). Hari anomali dingin tidak ditandai.
# Sengaja dipertahankan agar konsisten dengan dashboard; untuk dua sisi pakai (temp_mean_c - mean_per_city).abs() > 2 * std_per_city
df['is_anomaly_temp'] = (df['temp_mean_c']) > (2 * std_per_city + mean_per_city)

print('✅ T8 selesai')
print(f'   Hari hujan     : {df["is_rainy"].sum()} dari {len(df)} ({df["is_rainy"].mean()*100:.1f}%)')
print(f'   Extreme heat   : {df["is_extreme_heat"].sum()} hari')
print(f'   Anomali suhu   : {df["is_anomaly_temp"].sum()} hari')

✅ T8 selesai
   Hari hujan     : 192 dari 300 (64.0%)
   Extreme heat   : 7 hari
   Anomali suhu   : 5 hari


In [42]:
# ─────────────────────────────────────────────
# T9 — Deduplikasi & Final Check
# ─────────────────────────────────────────────

# Deduplikasi
before = len(df)
df = df.drop_duplicates(subset=['city_id', 'date'])
print(f'Deduplikasi: {before} → {len(df)} baris (hapus {before-len(df)})')

print('\n✅ T9 selesai — semua final checks lulus!')
print(f'\nDataFrame siap upload: {df.shape[0]} baris × {df.shape[1]} kolom')

# Preview kolom final
print(f'\nKolom final ({len(df.columns)}):')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2}. {col}')

Deduplikasi: 300 → 300 baris (hapus 0)

✅ T9 selesai — semua final checks lulus!

DataFrame siap upload: 300 baris × 34 kolom

Kolom final (34):
   1. city_id
   2. city_name
   3. province
   4. island
   5. date
   6. temp_max_c
   7. temp_min_c
   8. temp_mean_c
   9. precipitation_mm
  10. rain_mm
  11. wind_speed_kmh
  12. wind_gust_kmh
  13. humidity_pct
  14. uv_index
  15. weather_code
  16. sunshine_sec
  17. fetched_at
  18. year
  19. month
  20. month_name
  21. week
  22. day_name
  23. is_weekend
  24. season
  25. sunshine_hours
  26. temp_range_c
  27. heat_index_c
  28. heat_index_category
  29. weather_label
  30. rain_category
  31. uv_category
  32. is_rainy
  33. is_extreme_heat
  34. is_anomaly_temp


In [43]:
# ─────────────────────────────────────────────
# Rekonsiliasi dengan dashboard (snapshot 20 Sep – 19 Okt 2026)
# ─────────────────────────────────────────────
# Angka di bawah dibaca dari dashboard. Jika tidak cocok, data yang diproses bukan snapshot yang sama.
expected = {
    'Avg temperature (°C)'      : (round(df['temp_mean_c'].mean(), 1), 28.4),
    'Max temperature (°C)'      : (round(df['temp_max_c'].max(), 1), 38.1),
    'Total rain (mm)'           : (round(df['rain_mm'].sum()), 375),
    'Rainy city-days (%)'       : (round(df['is_rainy'].mean() * 100), 64),
    'Max UV index'              : (round(df['uv_index'].max(), 2), 9.65),
    'Avg daily max humidity (%)': (round(df['humidity_pct'].mean()), 87),
}
all_ok = True
for name, (got, dash) in expected.items():
    ok = abs(got - dash) < 1e-9
    all_ok &= ok
    print(f"{'✅' if ok else '⚠️'} {name:28} notebook = {got:<8} dashboard = {dash}")
print('\nSemua KPI cocok dengan dashboard ✅' if all_ok else '\n⚠️ Ada selisih: cek sumber data (USE_SNAPSHOT) dan tabel BigQuery.')


✅ Avg temperature (°C)         notebook = 28.4     dashboard = 28.4
✅ Max temperature (°C)         notebook = 38.1     dashboard = 38.1
✅ Total rain (mm)              notebook = 375      dashboard = 375
✅ Rainy city-days (%)          notebook = 64       dashboard = 64
✅ Max UV index                 notebook = 9.65     dashboard = 9.65
✅ Avg daily max humidity (%)   notebook = 87       dashboard = 87

Semua KPI cocok dengan dashboard ✅


---
## ☁️ STEP 3 — LOAD: Upload ke BigQuery

### Schema BigQuery yang Digunakan

```
Dataset: weather_dwh
    │
    ├── fact_weather_daily   ← tabel utama (1 baris per kota per hari)
    ├── dim_city             ← dimensi kota (metadata statis)
    └── vw_dashboard         ← view siap Looker Studio (join fact + dim)
```

### Kenapa 2 Tabel + 1 View?
- **fact_weather_daily** menyimpan semua measures (suhu, hujan, dll)
- **dim_city** menyimpan metadata kota yang jarang berubah (koordinat, provinsi, pulau)
- **vw_dashboard** menggabungkan keduanya agar Looker Studio tidak perlu JOIN manual

### Tips Koneksi BigQuery
1. Buat GCP project baru (gratis)
2. Aktifkan BigQuery API di GCP Console
3. Buat Service Account → role `BigQuery Data Editor` + `BigQuery Job User`
4. Download JSON key → taruh di path yang aman
5. Simpan `PROJECT_ID` dan `CREDENTIAL_PATH` di file `.env` (lihat `.env.example`). Jangan commit `.env` maupun file JSON key ke GitHub
6. Tanpa billing (BigQuery Sandbox), tabel kedaluwarsa otomatis; simpan salinan data di `data/`

In [44]:
# ─────────────────────────────────────────────
# TASK 3.1 — Inisialisasi BigQuery credentials
# ─────────────────────────────────────────────
# Dibaca dari file .env: PROJECT_ID dan CREDENTIAL_PATH (path file JSON service account).
# Jangan pernah commit .env atau file JSON key ke GitHub.
GCP_PROJECT_ID = get_secret('PROJECT_ID')
BQ_DATASET     = 'weather_dwh'

CREDENTIAL_PATH = get_secret('CREDENTIAL_PATH')
SCOPES = ["https://www.googleapis.com/auth/bigquery"]
credentials = service_account.Credentials.from_service_account_file(filename=CREDENTIAL_PATH,
                                                                    scopes=SCOPES)

In [45]:
if not LOAD_TO_BQ:
    print('⏭️  LOAD_TO_BQ = False → langkah ini dilewati (tabel BigQuery tidak diubah).')
else:
    # ─────────────────────────────────────────────
    # TASK 3.2 — Build & Load dim_city
    # ─────────────────────────────────────────────
    # dim_city berisi metadata statis kota.
    # Di-load dengan if_exists='replace' karena tidak berubah.

    df_dim_city = pd.DataFrame(CITIES)[['city_id','name','province','island','lat','lon']]\
        .rename(columns={'name':'city_name', 'lat':'latitude', 'lon':'longitude'})

    # Load to BigQuery
    TABLE_NAME = 'dim_city'
    pandas_gbq.to_gbq(
        dataframe=df_dim_city.drop_duplicates(),
        destination_table=f'{BQ_DATASET}.{TABLE_NAME}',
        project_id=GCP_PROJECT_ID,
        credentials=credentials,
        if_exists='replace'
    )

    print(f'✅ dim_city loaded — {len(df_dim_city)} kota')
    display(df_dim_city)


⏭️  LOAD_TO_BQ = False → langkah ini dilewati (tabel BigQuery tidak diubah).


In [46]:
if not LOAD_TO_BQ:
    print('⏭️  LOAD_TO_BQ = False → langkah ini dilewati (tabel BigQuery tidak diubah).')
else:
    # ─────────────────────────────────────────────
    # TASK 3.3 — Load fact_weather_daily
    # ─────────────────────────────────────────────
    # Strategi: REPLACE (bukan append) untuk idempotent.
    # Artinya: jalankan pipeline 2× = hasil sama, tidak ada duplikat.
    #
    # CATATAN: Untuk production dengan data lebih besar,
    # gunakan strategi DELETE partition + INSERT.

    # Siapkan kolom untuk BigQuery
    # Konversi date → datetime (BigQuery lebih konsisten dengan TIMESTAMP)
    df_load = df.copy()
    df_load['date'] = pd.to_datetime(df_load['date'])

    # Load to BigQuery
    TABLE_NAME = 'fact_weather_daily'
    pandas_gbq.to_gbq(
        dataframe=df_load.drop_duplicates(),
        destination_table=f'{BQ_DATASET}.{TABLE_NAME}',
        project_id=GCP_PROJECT_ID,
        credentials=credentials,
        if_exists='replace'
    )

    print(f'\n✅ fact_weather_daily loaded')
    print(f'   Baris : {len(df_load)}')
    print(f'   Kolom : {len(df_load.columns)}')


⏭️  LOAD_TO_BQ = False → langkah ini dilewati (tabel BigQuery tidak diubah).


In [47]:
if not LOAD_TO_BQ:
    print('⏭️  LOAD_TO_BQ = False → langkah ini dilewati (tabel BigQuery tidak diubah).')
else:
    # ─────────────────────────────────────────────
    # TASK 3.4 — Buat View untuk Looker Studio
    # ─────────────────────────────────────────────
    # View ini JOIN fact + dim_city sehingga Looker Studio tidak perlu konfigurasi JOIN manual.
    # Kolom ditulis eksplisit (bukan SELECT *) supaya tidak ada kolom ganda (mis. city_name ambigu).

    VIEW_SQL = f"""
    CREATE OR REPLACE VIEW `{GCP_PROJECT_ID}.{BQ_DATASET}.vw_dashboard` AS
    SELECT
        fwd.city_id, fwd.city_name, fwd.province, fwd.island, fwd.date,
        fwd.temp_max_c, fwd.temp_min_c, fwd.temp_mean_c,
        fwd.precipitation_mm, fwd.rain_mm,
        fwd.wind_speed_kmh, fwd.wind_gust_kmh,
        fwd.humidity_pct, fwd.uv_index, fwd.weather_code, fwd.sunshine_sec, fwd.fetched_at,
        fwd.year, fwd.month, fwd.month_name, fwd.week, fwd.day_name, fwd.is_weekend, fwd.season,
        fwd.sunshine_hours, fwd.temp_range_c, fwd.heat_index_c, fwd.heat_index_category,
        fwd.weather_label, fwd.rain_category, fwd.uv_category,
        fwd.is_rainy, fwd.is_extreme_heat, fwd.is_anomaly_temp,
        dc.latitude, dc.longitude
    FROM `{GCP_PROJECT_ID}.{BQ_DATASET}.fact_weather_daily` fwd
    JOIN `{GCP_PROJECT_ID}.{BQ_DATASET}.dim_city` dc
      ON fwd.city_id = dc.city_id
    """

    bq_client = bigquery.Client(project=GCP_PROJECT_ID, credentials=credentials)
    bq_client.query(VIEW_SQL).result()
    print('✅ vw_dashboard dibuat / diperbarui')


⏭️  LOAD_TO_BQ = False → langkah ini dilewati (tabel BigQuery tidak diubah).


---
## 🔍 STEP 4 — ANALYZE: Query Analitik di BigQuery

6 query berikut dirancang untuk menjawab pertanyaan bisnis yang sudah didefinisikan di awal.
Setiap query ekuivalen dengan satu komponen dashboard (dashboard sendiri membaca view `vw_dashboard`).

| Query | Pertanyaan | Chart di Looker Studio |
|---|---|---|
| Q1 | KPI utama per kota | Scorecard cards |
| Q2 | Tren suhu harian | Time series line chart |
| Q3 | Ranking kota | Horizontal bar chart |
| Q4 | Distribusi kondisi cuaca | Pie / Donut chart |
| Q5 | Heatmap hari dalam seminggu | Pivot table (heatmap) |
| Q6 | Anomali & hari ekstrem | Table dengan conditional formatting |

In [48]:
# ─────────────────────────────────────────────
# Q1 — KPI Summary per Kota
# ─────────────────────────────────────────────
# Digunakan untuk: Scorecard / KPI Cards di Looker Studio
# Satu baris per kota, berisi metrik ringkasan 30 hari.

# Helper untuk menjalankan query analitik (dipakai Q1-Q6)
VIEW = f'`{GCP_PROJECT_ID}.{BQ_DATASET}.vw_dashboard`'

def run_query(sql: str) -> pd.DataFrame:
    return pandas_gbq.read_gbq(sql, project_id=GCP_PROJECT_ID, credentials=credentials)

q1 = f"""
SELECT
    city_name, island,
    ROUND(AVG(temp_mean_c), 1)  AS avg_temp_c,
    MAX(temp_max_c)             AS max_temp_c,
    ROUND(SUM(rain_mm), 1)      AS total_rain_mm,
    COUNTIF(is_rainy)           AS rainy_days,
    COUNT(*)                    AS total_days,
    MAX(uv_index)               AS max_uv,
    ROUND(AVG(humidity_pct), 1) AS avg_daily_max_humidity_pct   -- rata-rata dari humidity MAKSIMUM harian
FROM {VIEW}
GROUP BY city_name, island
ORDER BY avg_temp_c DESC
"""
df_q1 = run_query(q1)
df_q1

Downloading: 100%|██████████|


,city_name,island,avg_temp_c,max_temp_c,total_rain_mm,rainy_days,total_days,max_uv,avg_daily_max_humidity_pct
0,Semarang,Jawa,29.9,38.1,16.3,20,30,9.40,80.2
1,Palembang,Sumatera,29.7,36.9,25.9,22,30,9.30,90.9
2,Jakarta,Jawa,29.6,37.2,57.0,29,30,9.40,86.4
3,Makassar,Sulawesi,29.5,35.8,1.4,4,30,9.40,76.6
4,Surabaya,Jawa,29.4,35.0,1.7,6,30,9.45,79.1
5,Balikpapan,Kalimantan,28.7,33.4,43.8,23,30,9.15,90.2
6,Yogyakarta,Jawa,27.8,37.4,14.2,18,30,9.45,87.6
7,Medan,Sumatera,26.6,33.2,167.1,30,30,9.10,98.4
8,Denpasar,Bali,26.6,31.7,9.4,17,30,9.35,94.9
9,Bandung,Jawa,26.0,35.1,38.6,23,30,9.65,85.0


In [49]:
# ─────────────────────────────────────────────
# Q2 — Tren Suhu Harian (Time Series)
# ─────────────────────────────────────────────
# Digunakan untuk: Line chart di Looker Studio
# Dimension: date, Metric: temp_mean_c, Breakdown: city_name
q2 = f"""
SELECT
    DATE(date) AS date,
    city_name,
    temp_mean_c,
    ROUND(AVG(temp_mean_c) OVER (
        PARTITION BY city_id ORDER BY DATE(date)
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 2) AS temp_7d_moving_avg
FROM {VIEW}
ORDER BY city_name, date
"""
df_q2 = run_query(q2)
df_q2.head()

Downloading: 100%|██████████|


,date,city_name,temp_mean_c,temp_7d_moving_avg
0,2026-09-20,Balikpapan,28.7,28.70
1,2026-09-21,Balikpapan,29.0,28.85
2,2026-09-22,Balikpapan,29.0,28.90
3,2026-09-23,Balikpapan,29.2,28.98
4,2026-09-24,Balikpapan,29.4,29.06


In [50]:
# ─────────────────────────────────────────────
# Q3 — Ranking Kota: Panas vs Hujan
# ─────────────────────────────────────────────
# Digunakan untuk: Horizontal bar chart di Looker Studio
# Satu baris per kota dengan ranking masing-masing metrik.
q3 = f"""
SELECT
    city_name,
    ROUND(AVG(temp_mean_c), 2) AS avg_temp_c,
    ROUND(SUM(rain_mm), 1)     AS total_rain_mm,
    RANK() OVER (ORDER BY AVG(temp_mean_c) DESC) AS heat_rank,
    RANK() OVER (ORDER BY SUM(rain_mm) DESC)     AS rain_rank
FROM {VIEW}
GROUP BY city_name
ORDER BY heat_rank
"""
df_q3 = run_query(q3)
df_q3

Downloading: 100%|██████████|


,city_name,avg_temp_c,total_rain_mm,heat_rank,rain_rank
0,Semarang,29.92,16.3,1,6
1,Palembang,29.68,25.9,2,5
2,Jakarta,29.56,57.0,3,2
3,Makassar,29.51,1.4,4,10
4,Surabaya,29.44,1.7,5,9
5,Balikpapan,28.71,43.8,6,3
6,Yogyakarta,27.80,14.2,7,7
7,Medan,26.65,167.1,8,1
8,Denpasar,26.55,9.4,9,8
9,Bandung,25.98,38.6,10,4


In [51]:
# ─────────────────────────────────────────────
# Q4 — Distribusi Kondisi Cuaca
# ─────────────────────────────────────────────
# Digunakan untuk: Pie / Donut chart di Looker Studio
# Tampilkan berapa persen hari cerah vs hujan vs mendung, dll.
q4 = f"""
SELECT
    rain_category,
    COUNT(*) AS day_count,
    ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM {VIEW}
GROUP BY rain_category
ORDER BY day_count DESC
"""
df_q4 = run_query(q4)
df_q4

Downloading: 100%|██████████|


,rain_category,day_count,pct
0,Light,175,58.3
1,No Rain,108,36.0
2,Moderate,17,5.7


In [52]:
# ─────────────────────────────────────────────
# Q5 — Heatmap: Suhu per Kota × Hari dalam Seminggu
# ─────────────────────────────────────────────
# Digunakan untuk: Pivot Table dengan heatmap coloring di Looker Studio
# Rows: city_name, Columns: day_name, Values: avg_temp_c
# Menjawab: "Hari apa yang cenderung terpanas di setiap kota?"
# PENTING: pakai AVG (bukan SUM) — dalam 30 hari, Senin & Minggu muncul 5x sedangkan hari lain 4x.
q5 = f"""
SELECT
    city_name,
    FORMAT_DATE('%u', DATE(date)) AS day_num,      -- 1=Senin ... 7=Minggu (untuk urutan kolom)
    day_name,
    ROUND(AVG(temp_mean_c), 2)    AS avg_temp_c,
    COUNT(*)                      AS day_count
FROM {VIEW}
GROUP BY city_name, day_num, day_name
ORDER BY city_name, day_num
"""
df_q5 = run_query(q5)
df_q5.pivot(index='city_name', columns='day_name', values='avg_temp_c')

Downloading: 100%|██████████|


day_name,Friday,Monday,Saturday,Sunday,Thursday,Tuesday,Wednesday
city_name,,,,,,,
Balikpapan,28.57,28.84,28.45,29.00,28.75,28.38,28.85
Bandung,25.83,25.98,25.77,25.60,26.15,26.48,26.15
Denpasar,26.50,26.58,26.70,26.42,26.55,26.53,26.60
Jakarta,29.63,29.30,29.98,29.42,29.45,29.93,29.32
Makassar,29.40,29.48,29.68,29.54,29.28,29.68,29.52
Medan,26.65,26.76,26.25,26.52,26.73,27.18,26.45
Palembang,29.73,29.80,29.55,29.60,29.77,29.70,29.63
Semarang,29.93,30.00,29.90,29.78,29.95,29.97,29.92
Surabaya,29.50,29.54,29.52,29.34,29.57,29.18,29.40


In [53]:
# ─────────────────────────────────────────────
# Q6 — Hari Anomali & Ekstrem
# ─────────────────────────────────────────────
# Digunakan untuk: Table dengan highlight di Looker Studio
# Tampilkan hari-hari yang memiliki kondisi tidak normal.
q6 = f"""
SELECT
    DATE(date) AS date,
    city_name,
    temp_mean_c,
    temp_max_c,
    rain_mm,
    CASE
        WHEN is_anomaly_temp AND is_extreme_heat THEN 'Heat Anomaly + Extreme Heat'
        WHEN is_anomaly_temp                     THEN 'Heat Anomaly'
        ELSE 'Extreme Heat'
    END AS alert_type
FROM {VIEW}
WHERE is_anomaly_temp OR is_extreme_heat
ORDER BY date, city_name
"""
df_q6 = run_query(q6)
df_q6

Downloading: 100%|██████████|


,date,city_name,temp_mean_c,temp_max_c,rain_mm,alert_type
0,2026-09-29,Medan,27.9,32.8,2.2,Heat Anomaly
1,2026-10-06,Semarang,30.9,37.3,0.0,Extreme Heat
2,2026-10-11,Makassar,30.8,35.5,0.0,Heat Anomaly
3,2026-10-12,Yogyakarta,29.0,35.6,0.0,Heat Anomaly
4,2026-10-13,Jakarta,30.8,37.2,2.7,Extreme Heat
5,2026-10-14,Semarang,30.2,37.5,0.0,Extreme Heat
6,2026-10-14,Yogyakarta,28.5,37.4,0.0,Extreme Heat
7,2026-10-15,Semarang,30.6,37.3,0.0,Extreme Heat
8,2026-10-16,Semarang,30.5,37.3,0.0,Extreme Heat
9,2026-10-19,Denpasar,27.6,31.7,0.7,Heat Anomaly


---
## 📊 STEP 5 — VISUALIZE: Looker Studio

**Dashboard:** https://datastudio.google.com/reporting/52b3495a-3735-4fb5-854e-51de1121a6dd

### Cara Koneksi BigQuery → Looker Studio

1. Buka **[lookerstudio.google.com](https://lookerstudio.google.com)**
2. Klik **+ Create** → **Report**
3. Pilih connector: **BigQuery**
4. Navigasi: `[project_id]` → `weather_dwh` → **`vw_dashboard`**
5. Klik **Connect** → **Add to Report**

---

### Layout Dashboard yang Direkomendasikan

```
┌─────────────────────────────────────────────────────────────────────┐
│  🌦️  Indonesia Weather Dashboard — Snapshot 20 Sep–19 Okt 2026      │
│  [Date Range Control]              [City Filter]  [Island Filter]   │
├──────────┬──────────┬──────────┬──────────┬──────────┬─────────────┤
│  AVG     │  MAX     │  TOTAL   │  RAINY   │  MAX UV  │  AVG DAILY  │
│  SUHU    │  SUHU    │  HUJAN   │  CITY-   │  INDEX   │  MAX        │
│          │          │          │  DAYS    │          │  HUMIDITY   │
│  28,4°C  │  38,1°C  │  375mm   │  64%     │  9,65    │   87%       │
├──────────┴──────────┴──────────┴──────────┴──────────┴─────────────┤
│                                                                      │
│  📈 Tren Suhu Harian per Kota (Line Chart)                           │
│  Dimension: date  |  Metric: temp_mean_c  |  Breakdown: city_name   │
│  Tambahkan: temp_7d_moving_avg sebagai line kedua                   │
│                                                                      │
├─────────────────────────────┬────────────────────────────────────────┤
│ 🏙️ Bar: Ranking Kota        │ 🌧️ Bar: Total Hujan per Kota           │
│  city_name × avg_temp_c     │  city_name × total_rain_mm             │
│  (urutkan descending)       │  (warna biru gradasi)                  │
├─────────────────────────────┼────────────────────────────────────────┤
│ 🍩 Donut: Kondisi Cuaca     │ 🔥 Pivot Heatmap: Suhu per Hari        │
│  rain_category × day_count  │  Rows: city_name                       │
│  (5 warna: No Rain→Extreme) │  Cols: day_name (Sen–Min)              │
│                             │  Values: avg_temp  (heatmap color)     │
├─────────────────────────────┴────────────────────────────────────────┤
│ ⚠️  Tabel: Hari Anomali & Ekstrem                                     │
│  Kolom: date, city_name, temp_mean_c, rain_mm, alert_type           │
│  Filter: alert_type ≠ 'Normal'  |  Conditional formatting on temp   │
└──────────────────────────────────────────────────────────────────────┘
```

### Konfigurasi KPI Scorecard

| Scorecard | Field | Aggregation | Format |
|---|---|---|---|
| Avg Suhu | `temp_mean_c` | Average | `##.#°C` |
| Max Suhu | `temp_max_c` | Max | `##.#°C` |
| Total Hujan | `rain_mm` | Sum | `###mm` |
| Rainy city-days (%) | `is_rainy` | hari-kota hujan ÷ total hari-kota (192/300) | `##%` |
| Max UV | `uv_index` | Max | `##.#` |
| Avg Daily Max Humidity | `humidity_pct` | Average (nilai = kelembaban maksimum harian) | `##%` |

### Filter yang Direkomendasikan

| Filter | Field | Tipe |
|---|---|---|
| Rentang tanggal | `date` | Date Range |
| Pilih kota | `city_name` | Dropdown (multi) |
| Pulau | `island` | Dropdown |
| Hanya hari hujan | `is_rainy` | Checkbox |
| Heat index | `heat_index_category` | Dropdown |

---
## 🔎 Temuan Utama (snapshot 20 Sep – 19 Okt 2026)

| Pertanyaan | Temuan |
|---|---|
| Kota terpanas / terdingin | **Semarang** (29,92 °C rata-rata) vs **Bandung** (25,98 °C): selisih ±3,9 °C |
| Suhu ekstrem | Suhu maksimum **38,1 °C** (Semarang, 19 Okt). **5 dari 7** hari *extreme heat* (suhu maks > 37 °C) terjadi di Semarang |
| Kota terhujan | **Medan** 167,1 mm ≈ **45%** dari total 375 mm (hujan tercatat 30 dari 30 hari). Makassar (1,4 mm) dan Surabaya (1,7 mm) paling kering |
| Intensitas hujan | 36,0% hari-kota tanpa hujan, 58,3% ringan, 5,7% sedang; tidak ada hujan lebat/ekstrem di periode ini |
| Hari terpanas dalam seminggu | Selisih rata-rata antar hari **< 1 °C** di semua kota, dengan hanya 4–5 observasi per sel, sehingga **tidak cukup bukti** adanya efek hari |
| Anomali | 5 hari anomali panas (> mean + 2σ per kota) dan 7 hari extreme heat |

**Catatan interpretasi:** periode hanya 30 hari dan data berasal dari model forecast, sehingga temuan bersifat deskriptif untuk periode ini, bukan klaim tentang iklim. Perluasan ke ≥ 12 bulan dan validasi terhadap data stasiun (mis. BMKG) diperlukan sebelum dipakai untuk keputusan.


---
## 📌 Catatan & Referensi

### Ringkasan Kolom Final

| Grup | Kolom |
|---|---|
| **Identitas** | city_id, city_name, province, island, date |
| **Suhu** | temp_max_c, temp_min_c, temp_mean_c, temp_range_c |
| **Feels Like** | heat_index_c, heat_index_category |
| **Hujan** | rain_mm, precipitation_mm, is_rainy, rain_category |
| **Angin** | wind_speed_kmh, wind_gust_kmh |
| **Lain** | humidity_pct, uv_index, uv_category, sunshine_hours |
| **Cuaca** | weather_code, weather_label |
| **Waktu** | year, month, month_name, week, day_name, is_weekend, season |
| **Flag** | is_extreme_heat, is_anomaly_temp (hanya anomali panas) |

### Referensi
- OpenMeteo API Docs: https://open-meteo.com/en/docs
- WMO Weather Codes: https://open-meteo.com/en/docs#weathervariables
- BigQuery Standard SQL: https://cloud.google.com/bigquery/docs/reference/standard-sql
- Looker Studio Help: https://support.google.com/looker-studio
- Heat Index Formula: Steadman (1979), NOAA adaptation